# Session 1 · Transformers and the Attention Mechanism

Generative AI › Track 4 · AI Architecture and Agents

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-01_attention_numpy.py`

In [1]:
"""
Session 1 · Example 2 — Scaled dot-product self-attention, step by step in NumPy.

Self-attention lets every word look at every other word and decide how much
each one matters for its own meaning:
    scores  = Q · K^T / sqrt(d_k)      (how well each query matches each key)
    weights = softmax(scores)          (each row sums to 1)
    output  = weights · V              (a weighted mix of the value vectors)

Run:  python session-01_attention_numpy.py
"""
import numpy as np

np.set_printoptions(precision=2, suppress=True)
rng = np.random.default_rng(7)

words = ["the", "cat", "sat", "down"]
d_model, d_k = 6, 4

# 1) Toy embeddings: one row per word (in a real model these are learned)
X = rng.normal(size=(len(words), d_model))

# 2) Learned projection matrices (random here) turn each embedding into Q, K, V
W_q = rng.normal(size=(d_model, d_k))
W_k = rng.normal(size=(d_model, d_k))
W_v = rng.normal(size=(d_model, d_k))
Q, K, V = X @ W_q, X @ W_k, X @ W_v
print("Shapes  X:", X.shape, " Q/K/V:", Q.shape)


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)   # subtract max for numerical stability
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


# 3) Raw scores, scaled by sqrt(d_k) so softmax does not saturate
scores = Q @ K.T / np.sqrt(d_k)
weights = softmax(scores)
output = weights @ V

print("\nAttention weights (row = word looking, column = word looked at)")
print("        " + "  ".join(f"{w:>5}" for w in words))
for w, row in zip(words, weights):
    print(f"{w:>6}  " + "  ".join(f"{v:5.2f}" for v in row))
print("\nEach row sums to:", weights.sum(axis=1))

for w, row in zip(words, weights):
    print(f"'{w}' attends most to '{words[int(row.argmax())]}' ({row.max():.0%})")

# 4) Why scale? Compare the sharpness of softmax with and without the sqrt(d_k)
big_d = 512
q = rng.normal(size=big_d); keys = rng.normal(size=(4, big_d))
raw = keys @ q
print(f"\nWith d_k={big_d}: unscaled max weight = {softmax(raw).max():.3f}, "
      f"scaled max weight = {softmax(raw / np.sqrt(big_d)).max():.3f}")

# 5) Causal (decoder) mask: a word may not look at future words
mask = np.triu(np.ones((4, 4)), k=1).astype(bool)
masked = np.where(mask, -1e9, scores)
print("\nCausal weights (upper triangle is zero):")
print(softmax(masked))
print("Output vectors shape:", output.shape)

Shapes  X: (4, 6)  Q/K/V: (4, 4)

Attention weights (row = word looking, column = word looked at)
          the    cat    sat   down
   the   0.32   0.52   0.14   0.02
   cat   0.00   0.00   0.00   1.00
   sat   0.17   0.75   0.08   0.00
  down   0.46   0.54   0.00   0.00

Each row sums to: [1. 1. 1. 1.]
'the' attends most to 'cat' (52%)
'cat' attends most to 'down' (100%)
'sat' attends most to 'cat' (75%)
'down' attends most to 'cat' (54%)

With d_k=512: unscaled max weight = 1.000, scaled max weight = 0.557

Causal weights (upper triangle is zero):
[[1.   0.   0.   0.  ]
 [0.93 0.07 0.   0.  ]
 [0.17 0.75 0.08 0.  ]
 [0.46 0.54 0.   0.  ]]
Output vectors shape: (4, 4)


### Example · `session-01_positional_encoding.py`

In [2]:
"""
Session 1 · Example 3 — Sinusoidal positional encoding and why word order needs it.

Attention by itself is order-blind: shuffling the words shuffles the outputs but
does not change them. Transformers add a position signal to each embedding:
    PE[pos, 2i]   = sin(pos / 10000^(2i/d))
    PE[pos, 2i+1] = cos(pos / 10000^(2i/d))

Run:  python session-01_positional_encoding.py
Creates: session-01_positional_encoding.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)


def positional_encoding(n_pos, d):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d // 2)[None, :]
    angle = pos / np.power(10000, 2 * i / d)
    pe = np.zeros((n_pos, d))
    pe[:, 0::2] = np.sin(angle)
    pe[:, 1::2] = np.cos(angle)
    return pe


pe = positional_encoding(50, 16)
print("PE shape:", pe.shape)
print("Position 0:", pe[0, :6])
print("Position 1:", pe[1, :6])

# 1) Order-blindness demo: plain self-attention (Q=K=V=X) on two word orders
def self_attend(X):
    s = X @ X.T / np.sqrt(X.shape[1])
    w = np.exp(s - s.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    return w @ X

rng = np.random.default_rng(0)
emb = {w: rng.normal(size=16) for w in ["dog", "bites", "man"]}
a = np.stack([emb[w] for w in ["dog", "bites", "man"]])
b = np.stack([emb[w] for w in ["man", "bites", "dog"]])
out_a, out_b = self_attend(a), self_attend(b)
print("\nWithout positions, 'dog' gets the same vector in both sentences:",
      np.allclose(out_a[0], out_b[2]))
out_a, out_b = self_attend(a + pe[:3]), self_attend(b + pe[:3])
print("With positions added, the vectors differ:", not np.allclose(out_a[0], out_b[2]))

# 2) Nearby positions are more similar than distant ones
def cos(u, v): return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
print("\nCosine similarity of position 10 with:")
for p in [9, 11, 15, 30, 49]:
    print(f"  position {p:2d}: {cos(pe[10], pe[p]):.3f}")

plt.figure(figsize=(8, 4.2))
plt.imshow(pe.T, aspect="auto", cmap="inferno", origin="lower")
plt.colorbar(label="value")
plt.xlabel("Position in the sentence"); plt.ylabel("Embedding dimension")
plt.title("Sinusoidal positional encoding (50 positions x 16 dims)")
plt.tight_layout(); plt.savefig("session-01_positional_encoding.png", dpi=110)
print("\nSaved chart: session-01_positional_encoding.png")

PE shape: (50, 16)
Position 0: [0. 1. 0. 1. 0. 1.]
Position 1: [0.841 0.54  0.311 0.95  0.1   0.995]

Without positions, 'dog' gets the same vector in both sentences: True
With positions added, the vectors differ: True

Cosine similarity of position 10 with:
  position  9: 0.936
  position 11: 0.936
  position 15: 0.767
  position 30: 0.722
  position 49: 0.595



Saved chart: session-01_positional_encoding.png


### Example · `session-01_multihead_compare.py`

In [3]:
"""
Session 1 · Example 4 — Multi-head attention with Keras, and transformers vs RNNs vs CNNs.

Part A builds a real Keras MultiHeadAttention layer and a full encoder block.
Part B counts how many steps each architecture needs to connect two distant words.

Run:  python session-01_multihead_compare.py
"""
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(42)

# ---------- Part A: multi-head attention and an encoder block ----------
seq_len, d_model, heads = 10, 32, 4
x = np.random.default_rng(1).normal(size=(2, seq_len, d_model)).astype("float32")

mha = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)
out, scores = mha(x, x, return_attention_scores=True)
print("Input  (batch, tokens, d_model):", x.shape)
print("Output (batch, tokens, d_model):", tuple(out.shape))
print("Scores (batch, heads, tokens, tokens):", tuple(scores.shape))
print("Each head's rows sum to 1:", np.allclose(np.asarray(scores).sum(-1), 1, atol=1e-5))


def encoder_block(inputs, d_model=32, heads=4, d_ff=64, rate=0.1):
    """Self-attention + Add&Norm + feed-forward + Add&Norm (the original encoder layer)."""
    attn = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)(inputs, inputs)
    x = layers.LayerNormalization(epsilon=1e-6)(inputs + layers.Dropout(rate)(attn))
    ff = layers.Dense(d_ff, activation="relu")(x)
    ff = layers.Dense(d_model)(ff)
    return layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(rate)(ff))


inp = keras.Input(shape=(seq_len, d_model))
model = keras.Model(inp, encoder_block(inp), name="encoder_block")
print(f"\nEncoder block parameters: {model.count_params():,}")

# ---------- Part B: path length between word 1 and word n ----------
print("\nSteps for information to travel from the first to the last token")
print(f"{'tokens':>7} | {'RNN (sequential)':>16} | {'CNN k=3 layers':>14} | {'Transformer':>11}")
for n in [10, 100, 1000]:
    rnn = n - 1                      # one hidden-state update per token, in order
    cnn = int(np.ceil((n - 1) / 2))  # each kernel-3 layer widens the view by 2
    print(f"{n:>7} | {rnn:>16} | {cnn:>14} | {1:>11}")

Input  (batch, tokens, d_model): (2, 10, 32)
Output (batch, tokens, d_model): (2, 10, 32)
Scores (batch, heads, tokens, tokens): (2, 4, 10, 10)
Each head's rows sum to 1: True

Encoder block parameters: 8,544

Steps for information to travel from the first to the last token
 tokens | RNN (sequential) | CNN k=3 layers | Transformer
     10 |                9 |              5 |           1
    100 |               99 |             50 |           1
   1000 |              999 |            500 |           1


### Example · `session-01_project_pronoun_resolver.py`

In [4]:
"""
Session 1 · Mini project (full solution) — Resolve "it" with self-attention.

Real-life problem: a support bot reads "The laptop did not fit in the bag because
it was too big." Which noun is "it"? Humans use meaning; transformers use
attention. Here we hand-design small feature embeddings (size, container,
object...) so that the attention pattern is easy to read, then:
  1. compute self-attention for two sentences that differ by ONE word,
  2. report which noun "it" attends to,
  3. draw both attention maps as a heatmap PNG.

Run:  python session-01_project_pronoun_resolver.py
Creates: session-01_attention_heatmap.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Feature axes: [is_object, is_container, is_big, is_small, is_pronoun, is_function_word]
VOCAB = {
    "the": [0, 0, 0, 0, 0, 1], "laptop": [1, 0, .6, .2, 0, 0], "did": [0, 0, 0, 0, 0, 1],
    "not": [0, 0, 0, 0, 0, 1], "fit": [0, 0, 0, 0, 0, .5], "in": [0, 0, 0, 0, 0, 1],
    "bag": [.3, 1, .2, .6, 0, 0], "because": [0, 0, 0, 0, 0, 1], "it": [0, 0, 0, 0, 1, 0],
    "was": [0, 0, 0, 0, 0, 1], "too": [0, 0, 0, 0, 0, 1], "big": [0, 0, 1, 0, 0, 0],
    "small": [0, 0, 0, 1, 0, 0],
}
NOUNS = {"laptop", "bag"}


def contextual_embed(words):
    """Give 'it' the meaning of the adjective that describes it (a stand-in for lower layers)."""
    E = np.array([VOCAB[w] for w in words], dtype=float)
    adj = words[-1]
    E[words.index("it")] += np.array(VOCAB[adj]) * 2   # 'it ... was too big' -> it is big
    return E


def attention(E, W_q, W_k):
    s = (E @ W_q) @ (E @ W_k).T / np.sqrt(W_q.shape[1])
    w = np.exp(s - s.max(1, keepdims=True))
    return w / w.sum(1, keepdims=True)


# Query/key projections: a big pronoun looks for an object; a small pronoun for a container
W_q = np.array([[0, 0], [0, 0], [1.2, 0], [0, 1.2], [.5, .5], [-1, -1]], dtype=float)
W_k = np.array([[1.2, 0], [0, 1.2], [.4, 0], [0, .4], [0, 0], [-1, -1]], dtype=float)

base = "the laptop did not fit in the bag because it was too".split()
results = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, adj in zip(axes, ["big", "small"]):
    words = base + [adj]
    A = attention(contextual_embed(words), W_q, W_k)
    row = A[words.index("it")]
    noun_scores = {w: row[i] for i, w in enumerate(words) if w in NOUNS}
    winner = max(noun_scores, key=noun_scores.get)
    results[adj] = winner
    print(f"Sentence: {' '.join(words)}.")
    print("  attention from 'it' ->", ", ".join(f"{w}: {s:.2f}" for w, s in noun_scores.items()))
    print(f"  resolved: 'it' = the {winner}\n")
    ax.imshow(A, cmap="inferno")
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=60, fontsize=8)
    ax.set_yticks(range(len(words))); ax.set_yticklabels(words, fontsize=8)
    ax.set_title(f"'...it was too {adj}'  ->  it = {winner}")
plt.tight_layout(); plt.savefig("session-01_attention_heatmap.png", dpi=110)

assert results == {"big": "laptop", "small": "bag"}, results
print("Both sentences resolved correctly. Saved chart: session-01_attention_heatmap.png")

Sentence: the laptop did not fit in the bag because it was too big.
  attention from 'it' -> laptop: 0.61, bag: 0.13
  resolved: 'it' = the laptop

Sentence: the laptop did not fit in the bag because it was too small.
  attention from 'it' -> laptop: 0.06, bag: 0.68
  resolved: 'it' = the bag



Both sentences resolved correctly. Saved chart: session-01_attention_heatmap.png
